In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, RandomRotation
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

IMAGE_SHAPE = (96, 96)  # Smaller than 224x224 — faster, better for upscaled 32px images
BATCH_SIZE = 64

# ── Load CIFAR-10 ────────────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# ── Data pipeline ────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.image.resize(image, IMAGE_SHAPE)       # 32x32 → 96x96 (less interpolation distortion)
    image = preprocess_input(image)                    # ResNet50 ImageNet-style normalization
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
    .shuffle(10000) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)) \
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)


Data Augmentation

In [ ]:
# ── Data augmentation (applied to raw pixels, before preprocess_input) ──
data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

def preprocess(image, label):
    image = tf.image.resize(image, IMAGE_SHAPE)
    image = preprocess_input(image)          # normalization after resize
    return image, label

def preprocess_and_augment(image, label):
    image = tf.image.resize(image, IMAGE_SHAPE)
    image = data_augment(image, training=True)  # augment before normalization
    image = preprocess_input(image)
    return image, label

# Original training set
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
    .shuffle(10000) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

# Augmented copy of the training set
aug_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
    .map(preprocess_and_augment, num_parallel_calls=tf.data.AUTOTUNE) \
    .shuffle(10000) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

# Concatenate: doubled training set (original + augmented)
combined_train_ds = train_ds.concatenate(aug_ds)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)) \
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

In [ ]:
aug_ds.prefetch(buffer_size = tf.data.AUTOTUNE)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.uint8, name=None))>

In [ ]:

# ── Model ────────────────────────────────────────────────────────
base_model = ResNet50(
    input_shape=IMAGE_SHAPE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

classifier = tf.keras.Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(10, activation="softmax")
])

classifier.summary()

# ── Phase 1: Train head only ─────────────────────────────────────
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_head = classifier.fit(
    combined_train_ds,
    epochs=10,
    validation_data=test_ds,
    callbacks=[early_stopping]
)

# ── Phase 2: Fine-tune top ResNet50 layers ───────────────────────
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower lr to preserve pretrained weights
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = classifier.fit(
    combined_train_ds,
    epochs=10,
    validation_data=test_ds,
    callbacks=[early_stopping]
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 3, 3, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,851,274 (90.99 MB)

 Trainable params: 263,562 (1.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/10
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 269s 160ms/step - accuracy: 0.7506 - loss: 0.7341 - val_accuracy: 0.8323 - val_loss: 0.4836
Epoch 2/10
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 243s 153ms/step - accuracy: 0.7872 - loss: 0.6153 - val_accuracy: 0.8500 - val_loss: 0.4288
Epoch 3/10
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 244s 153ms/step - accuracy: 0.8015 - loss: 0.5655 - val_accuracy: 0.8535 - val_loss: 0.4276
Epoch 4/10
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 258s 151ms/step - accuracy: 0.8097 - loss: 0.5400 - val_accuracy: 0.8552 - val_loss: 0.4314
Epoch 5/10
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 267s 154ms/step - accuracy: 0.8165 - loss: 0.5223 - val_accuracy: 0.8606 - val_loss: 0.4120
Epoch 6/10
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 251s 157ms/step - accuracy: 0.8232 - loss: 0.5055 - val_accuracy: 0.8669 - val_loss: 0.4010
Epoch 7/10
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 254s 153ms/step - accuracy: 0.8280 - loss: 0.4897 - val_accuracy: 0.8657 - val_loss: 0.3927
Epoch 8/10
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 266s 155ms/step - ac